# VectorDrugMatch - final audit GPU runs (2026-11-17)

Runs the GCN part of the final audit: (A) convergence rerun of A_k4_L3 with a 200-epoch budget on the historical split, (B) one-factor development CV,
(C) top configurations + combination with 3 seeds, (final) the best GCN configuration trained on `final_train` with 5 seeds.
Everything is predeclared in `configs/final_audit_20261117_plan.json`. **The locked holdout is never read in this notebook** (it is evaluated once,
later, after the selection is frozen).

Code pinned to `0b3d225f176e8733a779b46ae73292b18f9852e5` on branch `exp/final-audit-20261117`. Outputs go to a NEW Drive folder `MyDrive/VDM_Outputs/final_audit_20261117/`; no earlier folder is touched.
Run cells top to bottom. Every run is resumable: after a disconnect, rerun Steps 1-4 and then the stage cell.

## Step 1 - GPU

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then run again.'
print(torch.cuda.get_device_name(0))

## Step 2 - Drive, code at the pinned commit, libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf VectorDrugMatch
!git clone -q -b exp/final-audit-20261117 https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
PIN = '0b3d225f176e8733a779b46ae73292b18f9852e5'
!git checkout -q $PIN
!git log -1 --oneline

In [ ]:
%pip install -q torch-geometric pyyaml scikit-learn pandas scipy joblib
import torch_geometric; print('PyG', torch_geometric.__version__)
try:
    import pyg_lib; print('pyg-lib', pyg_lib.__version__)
except Exception as e:
    print('pyg-lib not available (RGCN may be slower):', e)

## Step 3 - Frozen benchmark + the committed final-protocol files, verified against the committed SHA-256 manifest (stop on mismatch)

In [ ]:
import hashlib, json, os, shutil, subprocess
D = '/content/drive/MyDrive/VDM_Data/frozen_processed/primekg'
assert os.path.isdir(D), 'MyDrive/VDM_Data/frozen_processed/primekg is missing'
shutil.copytree(D, 'data/processed/primekg', dirs_exist_ok=True)
shutil.rmtree('data/processed/primekg/splits_final_20261117', ignore_errors=True)
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
want = {f['path']: f['sha256'] for f in man if f['path'].startswith('data/processed/primekg/') and f['path'].endswith(('.csv', '.npy', '.npz'))}
bad = [p for p, h in want.items() if hashlib.sha256(open(p, 'rb').read()).hexdigest() != h]
print(len(want), 'frozen files checked;', 'MISMATCH' if bad else 'FROZEN BENCHMARK VERIFIED', bad); assert not bad
P = 'data/processed/primekg/splits_final_20261117'
shutil.copytree('reports/final_audit_20261117/protocol_files', P)      # the exact locked files generated once (not regenerated: not byte-portable)
ref = json.load(open('reports/final_audit_20261117/protocol_manifest_committed.json'))
same = {k: hashlib.sha256(open(f'{P}/{k}', 'rb').read()).hexdigest() == v['sha256'] for k, v in ref['files'].items()}
same['holdout'] = hashlib.sha256(open(f'{P}/holdout_LOCKED.npz', 'rb').read()).hexdigest() == ref['holdout']['sha256']
print(same); assert all(same.values()), 'protocol files differ from the committed manifest - STOP'
print('FINAL PROTOCOL VERIFIED')

## Step 4 - Output folder on Drive (rerun after every reconnect)

In [ ]:
import os
N = 'final_audit_20261117'
O = f'/content/drive/MyDrive/VDM_Outputs/{N}'
os.makedirs(O, exist_ok=True); os.makedirs('outputs', exist_ok=True)
if not os.path.islink(f'outputs/{N}'):
    os.symlink(O, f'outputs/{N}')
print(os.path.realpath(f'outputs/{N}'))

## Stage A - convergence: A_k4_L3, 200 epochs, patience 20, historical split, 5 seeds (~5 min)

In [ ]:
!python scripts/final_audit/select_dev.py run A

## Stage B - one-factor development CV: 13 configurations x 5 folds, seed 42 (~60-90 min)

In [ ]:
!python scripts/final_audit/select_dev.py run B

## Stage C - top 3 + combination, seeds 21 and 52 added (~30-45 min). The configurations are derived from Stage B validation results only.

In [ ]:
!python scripts/final_audit/select_dev.py run C && cat outputs/final_audit_20261117/stage_C_plan.json

## Final - best GCN configuration (predeclared rule) on final_train, 5 seeds (~5 min)

In [ ]:
!python scripts/final_audit/select_dev.py run final

## Check device, then save a light zip (dev_cv checkpoints excluded; convergence and final checkpoints INCLUDED). Wait for `SAVED`.

In [ ]:
import json, glob, collections, os
from google.colab import drive
rc = [json.load(open(f)) for f in glob.glob(f'outputs/{N}/**/run_config.json', recursive=True)]
print(len(rc), 'runs |', dict(collections.Counter(str(r['gpu']) for r in rc)), '|', dict(collections.Counter(r['git_commit'][:10] for r in rc)))
assert all(r['gpu'] for r in rc), 'some runs did not use the GPU'
Z = f'/content/drive/MyDrive/VDM_Outputs/{N}_results_light.zip'
!cd outputs && rm -f {Z} && zip -rq {Z} {N} -x '{N}/dev_cv/*/model.pth'
print('light zip MB:', round(os.path.getsize(Z) / 1e6, 1))
drive.flush_and_unmount()
print('SAVED - download', N + '_results_light.zip from MyDrive/VDM_Outputs')